In [52]:
# Importing libraries

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

In [54]:
# Loading preprocessed dataset

In [55]:
df = pd.read_csv(r"C:\Users\J.Shiva\OneDrive\Attachments\credit-risk-prediction\data\processed\loan_payments_preprocessed.csv")
print("Shape:", df.shape)
df.head()

Shape: (54231, 22)


,loan_amount,term,int_rate,instalment,grade,employment_length,home_ownership,annual_inc,verification_status,purpose,...,inq_last_6mths,open_accounts,total_accounts,out_prncp,total_payment,total_rec_prncp,total_rec_int,last_payment_amount,collections_12_mths_ex_med,target
0,8000,36,7.49,248.82,1,5,0,2.462697,0,1,...,0.526589,12,27,2.263644,2.197320,2.176171,1.979716,1.874988,0.0,0
1,13200,36,6.99,407.52,1,9,4,2.469776,0,1,...,0.000000,15,31,2.314163,2.250676,2.231980,2.037407,1.947700,0.0,0
2,16000,36,7.49,497.63,1,8,0,2.502309,1,1,...,0.000000,7,18,0.000000,2.373104,2.368411,2.043301,2.347672,0.0,0
3,15000,36,14.31,514.93,3,1,4,2.454915,1,2,...,0.000000,6,13,0.000000,2.368103,2.362350,2.061131,2.355148,0.0,0
4,15000,36,6.03,456.54,1,10,0,2.556025,2,2,...,0.526589,23,50,2.326277,2.262580,2.246657,2.034561,1.963731,0.0,0


In [56]:
# Splitting features and target

In [57]:
X = df.drop(columns=['target'])
y = df['target']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target distribution:\n", y.value_counts())

X shape: (54231, 21)
y shape: (54231,)
Target distribution:
 target
0    47289
1     6942
Name: count, dtype: int64


In [58]:
# Splitting data into train, validation and test sets

In [59]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)
print("Test shape:", X_test.shape)
print("\nTrain target distribution:\n", y_train.value_counts())

Train shape: (37983, 21)
Val shape: (8113, 21)
Test shape: (8135, 21)

Train target distribution:
 target
0    33121
1     4862
Name: count, dtype: int64


In [60]:
# Applying SMOTE on train set only

In [61]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", pd.Series(y_train_sm).value_counts().to_dict())

Before SMOTE: {0: 33121, 1: 4862}
After SMOTE: {0: 33121, 1: 33121}


In [62]:
# Training Logistic Regression model

In [63]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_sm, y_train_sm)

y_val_pred_lr = lr.predict(X_val)
print("Logistic Regression - Validation Results:")
print(classification_report(y_val, y_val_pred_lr))
print("ROC-AUC:", roc_auc_score(y_val, lr.predict_proba(X_val)[:,1]))

Logistic Regression - Validation Results:
              precision    recall  f1-score   support

           0       0.93      0.75      0.83      7074
           1       0.26      0.62      0.37      1039

    accuracy                           0.73      8113
   macro avg       0.60      0.68      0.60      8113
weighted avg       0.84      0.73      0.77      8113

ROC-AUC: 0.7438349383922417


In [64]:
# Training Random Forest model

In [65]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)

y_val_pred_rf = rf.predict(X_val)
print("Random Forest - Validation Results:")
print(classification_report(y_val, y_val_pred_rf))
print("ROC-AUC:", round(roc_auc_score(y_val, rf.predict_proba(X_val)[:,1]), 4))

Random Forest - Validation Results:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      7074
           1       1.00      0.82      0.90      1039

    accuracy                           0.98      8113
   macro avg       0.99      0.91      0.94      8113
weighted avg       0.98      0.98      0.98      8113

ROC-AUC: 0.9769


In [66]:
# Training XGBoost model

In [67]:
xgb = XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)
xgb.fit(X_train_sm, y_train_sm)

y_val_pred_xgb = xgb.predict(X_val)
print("XGBoost - Validation Results:")
print(classification_report(y_val, y_val_pred_xgb))
print("ROC-AUC:", round(roc_auc_score(y_val, xgb.predict_proba(X_val)[:,1]), 4))

XGBoost - Validation Results:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      7074
           1       0.98      0.85      0.91      1039

    accuracy                           0.98      8113
   macro avg       0.98      0.92      0.95      8113
weighted avg       0.98      0.98      0.98      8113

ROC-AUC: 0.9809


In [68]:
# Applying SMOTETomek on train set

In [69]:
from imblearn.combine import SMOTETomek

smotetomek = SMOTETomek(random_state=42)
X_train_st, y_train_st = smotetomek.fit_resample(X_train, y_train)

print("Before SMOTETomek:", y_train.value_counts().to_dict())
print("After SMOTETomek:", pd.Series(y_train_st).value_counts().to_dict())

Before SMOTETomek: {0: 33121, 1: 4862}
After SMOTETomek: {0: 32505, 1: 32505}


In [70]:
# Training Logistic Regression with SMOTETomek

In [71]:
lr_st = LogisticRegression(random_state=42, max_iter=1000)
lr_st.fit(X_train_st, y_train_st)

y_val_pred_lr_st = lr_st.predict(X_val)
print("Logistic Regression (SMOTETomek) - Validation Results:")
print(classification_report(y_val, y_val_pred_lr_st))
print("ROC-AUC:", round(roc_auc_score(y_val, lr_st.predict_proba(X_val)[:,1]), 4))

Logistic Regression (SMOTETomek) - Validation Results:
              precision    recall  f1-score   support

           0       0.93      0.72      0.81      7074
           1       0.25      0.65      0.36      1039

    accuracy                           0.71      8113
   macro avg       0.59      0.68      0.59      8113
weighted avg       0.85      0.71      0.75      8113

ROC-AUC: 0.7445


In [72]:
# Training Random Forest with SMOTETomek

In [73]:
rf_st = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_st.fit(X_train_st, y_train_st)

y_val_pred_rf_st = rf_st.predict(X_val)
print("Random Forest (SMOTETomek) - Validation Results:")
print(classification_report(y_val, y_val_pred_rf_st))
print("ROC-AUC:", round(roc_auc_score(y_val, rf_st.predict_proba(X_val)[:,1]), 4))

Random Forest (SMOTETomek) - Validation Results:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      7074
           1       1.00      0.82      0.90      1039

    accuracy                           0.98      8113
   macro avg       0.99      0.91      0.95      8113
weighted avg       0.98      0.98      0.98      8113

ROC-AUC: 0.9768


In [74]:
# Training XGBoost with SMOTETomek

In [75]:
xgb_st = XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)
xgb_st.fit(X_train_st, y_train_st)

y_val_pred_xgb_st = xgb_st.predict(X_val)
print("XGBoost (SMOTETomek) - Validation Results:")
print(classification_report(y_val, y_val_pred_xgb_st))
print("ROC-AUC:", round(roc_auc_score(y_val, xgb_st.predict_proba(X_val)[:,1]), 4))

XGBoost (SMOTETomek) - Validation Results:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      7074
           1       0.98      0.85      0.91      1039

    accuracy                           0.98      8113
   macro avg       0.98      0.92      0.95      8113
weighted avg       0.98      0.98      0.98      8113

ROC-AUC: 0.9827


In [76]:
# Applying Standard Scaling and SMOTE on train set

In [77]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_scaled_sm, y_train_scaled_sm = smote.fit_resample(X_train_scaled, y_train)

print("After Scaling + SMOTE:", pd.Series(y_train_scaled_sm).value_counts().to_dict())

After Scaling + SMOTE: {0: 33121, 1: 33121}


In [78]:
# Training Logistic Regression with scaling and SMOTE

In [79]:
lr_sc = LogisticRegression(random_state=42, max_iter=1000)
lr_sc.fit(X_train_scaled_sm, y_train_scaled_sm)

y_val_pred_lr_sc = lr_sc.predict(X_val_scaled)
print("Logistic Regression (Scaled + SMOTE) - Validation Results:")
print(classification_report(y_val, y_val_pred_lr_sc))
print("ROC-AUC:", round(roc_auc_score(y_val, lr_sc.predict_proba(X_val_scaled)[:,1]), 4))

Logistic Regression (Scaled + SMOTE) - Validation Results:
              precision    recall  f1-score   support

           0       0.98      0.94      0.96      7074
           1       0.67      0.85      0.75      1039

    accuracy                           0.93      8113
   macro avg       0.82      0.89      0.85      8113
weighted avg       0.94      0.93      0.93      8113

ROC-AUC: 0.9525


In [80]:
# Training Random Forest with scaling and SMOTE

In [81]:
rf_sc = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_sc.fit(X_train_scaled_sm, y_train_scaled_sm)

y_val_pred_rf_sc = rf_sc.predict(X_val_scaled)
print("Random Forest (Scaled + SMOTE) - Validation Results:")
print(classification_report(y_val, y_val_pred_rf_sc))
print("ROC-AUC:", round(roc_auc_score(y_val, rf_sc.predict_proba(X_val_scaled)[:,1]), 4))

Random Forest (Scaled + SMOTE) - Validation Results:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      7074
           1       1.00      0.83      0.90      1039

    accuracy                           0.98      8113
   macro avg       0.99      0.91      0.95      8113
weighted avg       0.98      0.98      0.98      8113

ROC-AUC: 0.9795


In [82]:
# Training XGBoost with scaling and SMOTE

In [83]:
xgb_sc = XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)
xgb_sc.fit(X_train_scaled_sm, y_train_scaled_sm)

y_val_pred_xgb_sc = xgb_sc.predict(X_val_scaled)
print("XGBoost (Scaled + SMOTE) - Validation Results:")
print(classification_report(y_val, y_val_pred_xgb_sc))
print("ROC-AUC:", round(roc_auc_score(y_val, xgb_sc.predict_proba(X_val_scaled)[:,1]), 4))

XGBoost (Scaled + SMOTE) - Validation Results:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      7074
           1       0.99      0.85      0.91      1039

    accuracy                           0.98      8113
   macro avg       0.99      0.92      0.95      8113
weighted avg       0.98      0.98      0.98      8113

ROC-AUC: 0.982


In [84]:
# Applying Standard Scaling and SMOTETomek on train set

In [85]:
X_train_scaled_st, y_train_scaled_st = smotetomek.fit_resample(X_train_scaled, y_train)

print("After Scaling + SMOTETomek:", pd.Series(y_train_scaled_st).value_counts().to_dict())

After Scaling + SMOTETomek: {0: 33094, 1: 33094}


In [86]:
# Training Logistic Regression with scaling and SMOTETomek

In [87]:
lr_st_sc = LogisticRegression(random_state=42, max_iter=1000)
lr_st_sc.fit(X_train_scaled_st, y_train_scaled_st)

y_val_pred_lr_st_sc = lr_st_sc.predict(X_val_scaled)
print("Logistic Regression (Scaled + SMOTETomek) - Validation Results:")
print(classification_report(y_val, y_val_pred_lr_st_sc))
print("ROC-AUC:", round(roc_auc_score(y_val, lr_st_sc.predict_proba(X_val_scaled)[:,1]), 4))

Logistic Regression (Scaled + SMOTETomek) - Validation Results:
              precision    recall  f1-score   support

           0       0.98      0.94      0.96      7074
           1       0.67      0.85      0.75      1039

    accuracy                           0.93      8113
   macro avg       0.83      0.90      0.85      8113
weighted avg       0.94      0.93      0.93      8113

ROC-AUC: 0.9525


In [88]:
# Training Random Forest with scaling and SMOTETomek

In [89]:
rf_st_sc = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_st_sc.fit(X_train_scaled_st, y_train_scaled_st)

y_val_pred_rf_st_sc = rf_st_sc.predict(X_val_scaled)
print("Random Forest (Scaled + SMOTETomek) - Validation Results:")
print(classification_report(y_val, y_val_pred_rf_st_sc))
print("ROC-AUC:", round(roc_auc_score(y_val, rf_st_sc.predict_proba(X_val_scaled)[:,1]), 4))

Random Forest (Scaled + SMOTETomek) - Validation Results:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      7074
           1       1.00      0.83      0.90      1039

    accuracy                           0.98      8113
   macro avg       0.99      0.91      0.95      8113
weighted avg       0.98      0.98      0.98      8113

ROC-AUC: 0.9806


In [90]:
# Training XGBoost with scaling and SMOTETomek

In [91]:
xgb_st_sc = XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)
xgb_st_sc.fit(X_train_scaled_st, y_train_scaled_st)

y_val_pred_xgb_st_sc = xgb_st_sc.predict(X_val_scaled)
print("XGBoost (Scaled + SMOTETomek) - Validation Results:")
print(classification_report(y_val, y_val_pred_xgb_st_sc))
print("ROC-AUC:", round(roc_auc_score(y_val, xgb_st_sc.predict_proba(X_val_scaled)[:,1]), 4))

XGBoost (Scaled + SMOTETomek) - Validation Results:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      7074
           1       0.99      0.85      0.91      1039

    accuracy                           0.98      8113
   macro avg       0.98      0.92      0.95      8113
weighted avg       0.98      0.98      0.98      8113

ROC-AUC: 0.9826


In [92]:
# Saving best model and scaler

In [93]:
import joblib

model_path = r"C:\Users\J.Shiva\OneDrive\Attachments\credit-risk-prediction\models\xgb_model.pkl"
scaler_path = r"C:\Users\J.Shiva\OneDrive\Attachments\credit-risk-prediction\models\scaler.pkl"

joblib.dump(xgb_sc, model_path)
joblib.dump(scaler, scaler_path)
print("Model and scaler saved successfully!")

Model and scaler saved successfully!


In [94]:
## Trying LightGBM Model

In [95]:
import lightgbm as lgb
print(lgb.__version__)

4.6.0


In [96]:
# Training LightGBM with scaling and SMOTE

In [97]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(random_state=42, n_jobs=-1)
lgbm.fit(X_train_scaled_sm, y_train_scaled_sm)

y_val_pred_lgbm = lgbm.predict(X_val_scaled)
print("LightGBM (Scaled + SMOTE) - Validation Results:")
print(classification_report(y_val, y_val_pred_lgbm))
print("ROC-AUC:", round(roc_auc_score(y_val, lgbm.predict_proba(X_val_scaled)[:,1]), 4))

[LightGBM] [Info] Number of positive: 33121, number of negative: 33121
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005001 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5111
[LightGBM] [Info] Number of data points in the train set: 66242, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
LightGBM (Scaled + SMOTE) - Validation Results:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      7074
           1       1.00      0.84      0.91      1039

    accuracy                           0.98      8113
   macro avg       0.99      0.92      0.95      8113
weighted avg       0.98      0.98      0.98      8113

ROC-AUC: 0.9813


In [98]:
# Training LightGBM with scaling and SMOTETomek

In [99]:
lgbm_st = LGBMClassifier(random_state=42, n_jobs=-1)
lgbm_st.fit(X_train_scaled_st, y_train_scaled_st)

y_val_pred_lgbm_st = lgbm_st.predict(X_val_scaled)
print("LightGBM (Scaled + SMOTETomek) - Validation Results:")
print(classification_report(y_val, y_val_pred_lgbm_st))
print("ROC-AUC:", round(roc_auc_score(y_val, lgbm_st.predict_proba(X_val_scaled)[:,1]), 4))

[LightGBM] [Info] Number of positive: 33094, number of negative: 33094
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004835 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5111
[LightGBM] [Info] Number of data points in the train set: 66188, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
LightGBM (Scaled + SMOTETomek) - Validation Results:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      7074
           1       1.00      0.84      0.91      1039

    accuracy                           0.98      8113
   macro avg       0.99      0.92      0.95      8113
weighted avg       0.98      0.98      0.98      8113

ROC-AUC: 0.9812


In [100]:
#summary

## Final Model Comparison - Validation Set

| Model | Sampling | Recall (Default) | Precision (Default) | F1 (Default) | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | SMOTE | 0.81 | 0.47 | 0.59 | 0.9100 |
| Logistic Regression | SMOTETomek | 0.65 | 0.25 | 0.36 | 0.7445 |
| Logistic Regression | Scaled + SMOTE | 0.85 | 0.67 | 0.75 | 0.9525 |
| Logistic Regression | Scaled + SMOTETomek | 0.85 | 0.67 | 0.75 | 0.9525 |
| Random Forest | SMOTE | 0.82 | 1.00 | 0.90 | 0.9766 |
| Random Forest | SMOTETomek | 0.82 | 1.00 | 0.90 | 0.9768 |
| Random Forest | Scaled + SMOTE | 0.83 | 1.00 | 0.90 | 0.9795 |
| Random Forest | Scaled + SMOTETomek | 0.83 | 1.00 | 0.90 | 0.9806 |
| LightGBM | Scaled + SMOTE | 0.84 | 1.00 | 0.91 | 0.9813 |
| LightGBM | Scaled + SMOTETomek | 0.84 | 1.00 | 0.91 | 0.9812 |
| XGBoost | SMOTE | 0.85 | 0.99 | 0.91 | 0.9797 |
| XGBoost | SMOTETomek | 0.85 | 0.98 | 0.91 | 0.9827 |
| XGBoost | Scaled + SMOTETomek | 0.85 | 0.99 | 0.91 | 0.9826 |
| **XGBoost** | **Scaled + SMOTE** | **0.85** | **0.99** | **0.91** | **0.9820** |

**Winner: XGBoost with Scaled + SMOTE** — highest ROC-AUC (0.9820), best overall balance